In [ ]:
SYSTEM_DIR = '/content/drive/MyDrive/Drive_monitoring/training_colab'
TRAIN_WINDOWS_PATH = '/content/drive/MyDrive/Drive_monitoring/feature_engineeringV3/outputs/20260717T160540Z/window_features_train.csv'
TEST_WINDOWS_PATH = '/content/drive/MyDrive/Drive_monitoring/feature_engineeringV3/outputs/20260717T160540Z/window_features_test.csv'
FEATURE_DECISIONS_PATH = '/content/drive/MyDrive/Drive_monitoring/feature_engineeringV3/outputs/20260717T160540Z/feature_decisions.csv'
OUTPUT_PATH = '/content/drive/MyDrive/Drive_monitoring/training_colab/dms_final_outputs'
RUN_MODE = 'smoke'  # smoke | full
SEEDS = [42, 43, 44]
FEATURE_COUNTS = [7, 20, 40, 65]
OPTUNA_TRIALS = 25  # per each of the two Stage-B feature lists
USE_GPU = True

## Setup: run the standalone training package directly from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys
from pathlib import Path

system_drive = Path(SYSTEM_DIR)
if not system_drive.is_dir():
    raise FileNotFoundError(f'training_colab folder was not found: {system_drive}')
for required in ('dms_training', 'tests', 'requirements.txt'):
    if not (system_drive / required).exists():
        raise FileNotFoundError(f'Missing inside training_colab: {required}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(system_drive / 'requirements.txt')])
system_drive_str = str(system_drive)
sys.path = [item for item in sys.path if item != system_drive_str]
sys.path.insert(0, system_drive_str)

# Run All may reuse the same Colab kernel. Remove an older in-memory copy.
import importlib
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == 'dms_training' or module_name.startswith('dms_training.'):
        del sys.modules[module_name]
from dms_training import OUTPUT_CONTRACT_VERSION
if OUTPUT_CONTRACT_VERSION != 2:
    raise RuntimeError(
        f'Wrong dms_training output contract: {OUTPUT_CONTRACT_VERSION}; expected 2. ' 
        'Replace the complete training_colab folder on Drive and restart the runtime.'
    )
print('Training package:', system_drive, '| output contract:', OUTPUT_CONTRACT_VERSION)

if USE_GPU:
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
    if probe.returncode != 0:
        raise RuntimeError('USE_GPU=True but no Colab GPU is visible. Select Runtime > Change runtime type > T4 GPU.')
    print('GPU:', probe.stdout.strip())
else:
    print('LightGBM CPU mode selected (recommended for this tabular workload).')

input_paths = {}
for key, source in {'train': TRAIN_WINDOWS_PATH, 'test': TEST_WINDOWS_PATH, 'decisions': FEATURE_DECISIONS_PATH}.items():
    source = Path(source)
    if not source.is_file():
        raise FileNotFoundError(source)
    input_paths[key] = source
    print(key, source, f'{source.stat().st_size / (1024**2):.1f} MB')

## Contract and unit tests

In [ ]:
env = dict(os.environ, PYTHONPATH=str(system_drive))
subprocess.check_call([
    sys.executable, '-m', 'pytest', '-q',
    str(system_drive / 'tests'),
], cwd=str(system_drive), env=env)

## Run smoke or full scientific protocol

In [ ]:
from dms_training.config import ExperimentConfig
from dms_training.experiment import run_experiment

experiment_config = ExperimentConfig(
    train_windows_path=input_paths['train'],
    test_windows_path=input_paths['test'],
    feature_decisions_path=input_paths['decisions'],
    output_path=Path(OUTPUT_PATH),
    run_mode=RUN_MODE,
    seeds=SEEDS,
    feature_counts=FEATURE_COUNTS,
    optuna_trials=OPTUNA_TRIALS,
    use_gpu=USE_GPU,
)
run_dir = run_experiment(experiment_config)
print('Completed:', run_dir)

## Final verification, comparison and ZIP handoff

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

required = [
    'comparison.csv', 'decision_report.md', 'oof_predictions.csv',
    'test_predictions.csv', 'metrics_overall.json', 'metrics_per_dataset.csv',
    'lodo_metrics.csv', 'threshold_sweep.csv', 'errors_fp_fn.csv', 'split_manifest.csv',
    'fit_audit.csv', 'execution_summary.json',
    'model_bundle/model.txt', 'model_bundle/calibration.json',
    'model_bundle/operating_point.json', 'model_bundle/feature_schema.json',
    'model_bundle/training_manifest.json', 'model_bundle/golden_samples.json',
    'model_bundle/checksums.sha256', 'model_bundle/MODEL_CARD.md',
]
audit_outputs = {'fit_audit.csv', 'execution_summary.json'}
missing = [name for name in required if not (run_dir / name).is_file()]
missing_core = [name for name in missing if name not in audit_outputs]
if missing_core:
    raise RuntimeError(f'Incomplete run; missing core outputs: {missing_core}')
if missing:
    print('WARNING: This run was produced by the legacy in-memory training package.')
    print('Missing audit-only outputs:', missing)
    print('The result will be zipped, but per-fit/GPU auditing is unavailable. Restart the runtime before the next run.')
else:
    execution = json.loads((run_dir / 'execution_summary.json').read_text())
    if not execution.get('complete', False):
        raise RuntimeError(f'Planned and actual fit counts differ: {execution}')
    display(pd.DataFrame([execution]))
display(pd.read_csv(run_dir / 'comparison.csv'))
display(Markdown((run_dir / 'decision_report.md').read_text()))
results_zip = shutil.make_archive(str(run_dir), 'zip', root_dir=run_dir)
bundle_zip = shutil.make_archive(str(run_dir / 'model_bundle'), 'zip', root_dir=run_dir / 'model_bundle')
print('Results ZIP:', results_zip)
print('Model Bundle ZIP:', bundle_zip)